#### Data Project 1: Transits 
#### Kenny Tran

#### We will start by importing the packages we need to carry on this project.


In [17]:
import os
import json
import numpy
import datetime
import certifi
import pandas as pd
import pymongo
import sqlalchemy
from sqlalchemy import create_engine, text

#### Next, we will declare & assign connection variables for the MongoDB server, the MySQL server & databases with which I will be working 

In [18]:
mysql_args = {
    "uid": "root",
    "pwd": "Lord!ken123!",     
    "hostname": "localhost",
    "dbname": "transit_dw"    
}

mongodb_args = {
    "user_name": "kennyttran03_db_user",
    "password": "eS08JBItmBAImVH4",
    "cluster_name": "kenny-cluster",
    "cluster_subnet": "7fi3uyt",
    "cluster_location": "atlas", 
    "db_name": "transit"
}

#### Now to establish the helper connection for getting data from and setting data into databases

In [19]:
def get_sql_dataframe(sql_query, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    '''Invoke the pd.read_sql() function to query the database, and fill a Pandas DataFrame.'''
    dframe = pd.read_sql(text(sql_query), connection);
    connection.close()
    
    return dframe
    

def set_dataframe(df, table_name, pk_column, db_operation, **args):
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    try:
        if db_operation == "insert":
            df.to_sql(table_name, con=connection, index=False, if_exists='replace')
            if pk_column:
                connection.execute(text(f"ALTER TABLE {table_name} ADD PRIMARY KEY ({pk_column});"))
        elif db_operation == "update":
            df.to_sql(table_name, con=connection, index=False, if_exists='append')
    finally:
        connection.close()


def get_mongo_client(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the cluster_location parameter.")
    
    else:
        if args["cluster_location"] == "atlas":
            connect_str = f"mongodb+srv://{args['user_name']}:{args['password']}@"
            connect_str += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net"
            client = pymongo.MongoClient(connect_str, tlsCAFile=certifi.where())
            
        elif args["cluster_location"] == "local":
            client = pymongo.MongoClient("mongodb://localhost:27017/")
        
    return client


def get_mongo_dataframe(mongo_client, db_name, collection, query=None, projection=None):
    """Query MongoDB and return a pandas DataFrame. Safely handles empty results and _id."""
    query = query or {}
    if projection is None:
        projection = {"_id": 0}

    db = mongo_client[db_name]
    docs = list(db[collection].find(query, projection))
    df = pd.DataFrame(docs)
    if "_id" in df.columns:
        df.drop(columns=["_id"], inplace=True)

    return df


def set_mongo_collections(mongo_client, db_name, data_directory, json_files):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()

In [20]:
sqlEngine = create_engine(f"mysql+pymysql://{mysql_args['uid']}:{mysql_args['pwd']}@{mysql_args['hostname']}/", pool_recycle=3600)
DB_NAME = mysql_args["dbname"]
connection=sqlEngine.connect()

connection.execute(text(f"DROP DATABASE IF EXISTS {DB_NAME};"))
connection.execute(text(f"CREATE DATABASE {DB_NAME};"))
connection.execute(text(f"Use {DB_NAME};"))
print(f"Dropped and created database: {DB_NAME}")

Dropped and created database: transit_dw


#### We first create `dim_routes` table in the transit_dw database and do the same with `dim_buses`, `dim_stops`, and `fact_riders` in addition to the staging tables `stg_riders`, `stg_routes`, `stg_stops`, and `stg_buses`
(`stg_riders`, `stg_routes`, and `stg_stops` are from CSV file source as per the requirements)


(`stg_buses` is the NoSQL (Mongo) source as per the requirements)

In [21]:
delete_dim_routes = """
DROP TABLE IF EXISTS dim_routes;
"""

create_dim_routes = """
CREATE TABLE IF NOT EXISTS dim_routes (
  route_key INT NOT NULL AUTO_INCREMENT PRIMARY KEY,
  route_id VARCHAR(32) NOT NULL,
  route_name VARCHAR(100),
  origin VARCHAR(100),
  destination VARCHAR(100),
  service_type VARCHAR(50),
  route_length_km DECIMAL(6,2),
  UNIQUE KEY (route_id)
);
"""
connection.execute(text(delete_dim_routes))
connection.execute(text(create_dim_routes))
print("Created table: dim_routes successfully.")

Created table: dim_routes successfully.


In [22]:
delete_dim_buses = """
DROP TABLE IF EXISTS dim_buses;
"""
create_dim_buses = """
CREATE TABLE IF NOT EXISTS dim_buses (
  bus_key INT NOT NULL AUTO_INCREMENT PRIMARY KEY,
  bus_id VARCHAR(32) NOT NULL,
  capacity INT,
  manufacturer VARCHAR(80),
  model VARCHAR(80),
  year SMALLINT,
  fuel_type VARCHAR(40),
  UNIQUE KEY (bus_id)
);
"""
connection.execute(text(delete_dim_buses))
connection.execute(text(create_dim_buses))
print("Created table: dim_buses successfully.")

Created table: dim_buses successfully.


In [23]:
delete_dim_stops = """
DROP TABLE IF EXISTS dim_stops;
"""
create_dim_stops = """
CREATE TABLE IF NOT EXISTS dim_stops (
  stop_key INT NOT NULL AUTO_INCREMENT PRIMARY KEY,
  stop_id VARCHAR(32) NOT NULL,
  stop_name VARCHAR(120),
  latitude DECIMAL(9,6),
  longitude DECIMAL(9,6),
  zone VARCHAR(40),
  UNIQUE KEY (stop_id)
);
"""
connection.execute(text(delete_dim_stops))
connection.execute(text(create_dim_stops))
print("Created table: dim_stops successfully.")

Created table: dim_stops successfully.


In [24]:
delete_fact_riders = """
DROP TABLE IF EXISTS fact_riders;
"""
create_fact_riders = """
CREATE TABLE IF NOT EXISTS fact_riders (
  ride_key BIGINT NOT NULL AUTO_INCREMENT PRIMARY KEY,
  route_id VARCHAR(32),
  bus_id VARCHAR(32),
  stop_id VARCHAR(32),
  ride_dt DATE,
  time_recorded TIME,
  riders_count INT,
  fare_collected DECIMAL(10,2),
  avg_wait_time DECIMAL(5,2),
  date_key INT,
  route_key INT,
  bus_key INT,
  stop_key INT,
  KEY (date_key), KEY (route_key), KEY (bus_key), KEY (stop_key)
);
"""
connection.execute(text(delete_fact_riders))
connection.execute(text(create_fact_riders))
print("Created table: fact_ridership successfully.")

Created table: fact_ridership successfully.


#### Now for the staging tables:

In [25]:
delete_stg_riders = """
DROP TABLE IF EXISTS stg_riders;
"""
create_stg_riders = """
CREATE TABLE stg_riders (
  ride_dt DATE,
  time_recorded TIME,
  route_id VARCHAR(32),
  stop_id VARCHAR(32),
  bus_id VARCHAR(32),
  riders_count INT,
  fare_collected DECIMAL(10,2),
  avg_wait_time DECIMAL(5,2)
);
"""
connection.execute(text(delete_stg_riders))
connection.execute(text(create_stg_riders))
print("Created table: stg_riders successfully.")

Created table: stg_riders successfully.


In [26]:
delete_stg_routes = """
DROP TABLE IF EXISTS stg_routes;
"""
create_stg_routes = """
CREATE TABLE IF NOT EXISTS stg_routes (
  route_id VARCHAR(32),
  route_name VARCHAR(100),
  origin VARCHAR(100),
  destination VARCHAR(100),
  service_type VARCHAR(50),
  route_length_km DECIMAL(6,2)
);
"""
connection.execute(text(delete_stg_routes))
connection.execute(text(create_stg_routes))
print("Created table: stg_routes successfully.")

Created table: stg_routes successfully.


In [27]:
delete_stg_stops = """
DROP TABLE IF EXISTS stg_stops;
"""
create_stg_stops = """
CREATE TABLE stg_stops (
  stop_id VARCHAR(32),
  stop_name VARCHAR(120),
  latitude DECIMAL(9,6),
  longitude DECIMAL(9,6),
  zone VARCHAR(40)
);
"""
connection.execute(text(delete_stg_stops))
connection.execute(text(create_stg_stops))
print("Created table: stg_stops successfully.")


Created table: stg_stops successfully.


Now for stg_buses for Mongo

In [28]:
delete_stg_buses = """
DROP TABLE IF EXISTS stg_buses;
"""
create_stg_buses = """
CREATE TABLE stg_buses (
  bus_id VARCHAR(32),
  capacity INT,
  manufacturer VARCHAR(80),
  model VARCHAR(80),
  year SMALLINT,
  fuel_type VARCHAR(40)
);
"""
connection.execute(text(delete_stg_buses))
connection.execute(text(create_stg_buses))
print("Created table: stg_buses successfully.")



Created table: stg_buses successfully.


#### Populate `dim_date` (Executed in MySQL Workbench)
The script `dim_date_create_transit.sql`were run in MySQL Workbench.

- `dim_date_create_transit.sql` created and populated the `dim_date` table using a stored procedure.

At this point `dim_date` is ready for integration. Now we run a command to check if the tables are present in the database:

In [29]:
get_sql_dataframe("SHOW TABLES;", **mysql_args)

,Tables_in_transit_dw
0,dim_buses
1,dim_date
2,dim_routes
3,dim_stops
4,fact_riders
5,stg_buses
6,stg_riders
7,stg_routes
8,stg_stops


 #### Now to seed the collection `buses` into MongoDB to initialize the NoSQL data source with starting data so my ETL pipeline can extract from it. 

In [30]:
client = get_mongo_client(**mongodb_args)
db = client[mongodb_args["db_name"]]

db.drop_collection("buses")
db.buses.insert_many([
    {"bus_id": "B-12", "capacity": 42, "manufacturer": "Gillig", "model": "Low Floor", "year": 2021, "fuel_type": "Diesel"},
    {"bus_id": "B-07", "capacity": 30, "manufacturer": "BYD", "model": "K7M", "year": 2022, "fuel_type": "Electric"}
])
print("Seeded 'Buses' MongoDB collection")

Seeded 'Buses' MongoDB collection


#### In this cell, We will be reading the csv files that are placed in the data folder (this folder will be attached to the submission)

In [31]:
mongo = get_mongo_client(**mongodb_args)
riders =pd.read_csv("data/daily_riders.csv", parse_dates=["ride_dt"])
routes= pd.read_csv("data/routes_catalog.csv")
stops = pd.read_csv("data/stops.csv")
routes.head()
stops.head()



,id,stop_name,latitude,longitude,zone,notes,attachments
0,S101,Downtown Hub,38.0293,-78.4767,A,none,none
1,S103,Hospital Entrance,38.0312,-78.5001,A,none,none
2,S205,Airport Terminal,38.1400,-78.4450,B,none,none


#### Now to utilize some transformations before loading and inserting the data into staging

In [32]:
drop_cols = ['notes', 'attachments']
routes.drop(drop_cols, axis=1, inplace=True)
stops.drop(drop_cols, axis=1, inplace=True)
routes.rename(columns={"id": "route_id"}, inplace=True)
stops.rename(columns={"id": "stop_id"}, inplace=True)
routes.head()
stops.head()

,stop_id,stop_name,latitude,longitude,zone
0,S101,Downtown Hub,38.0293,-78.4767,A
1,S103,Hospital Entrance,38.0312,-78.5001,A
2,S205,Airport Terminal,38.1400,-78.4450,B


#### After some transformations, we are not loading the data from csv and mongo collection into staging files 

In [33]:
set_dataframe(riders,"stg_riders", pk_column=None, db_operation="insert", **mysql_args)
set_dataframe(routes,"stg_routes",pk_column=None, db_operation="insert", **mysql_args)
set_dataframe(stops,"stg_stops", pk_column=None, db_operation="insert", **mysql_args)
buses_df = get_mongo_dataframe(mongo, mongodb_args["db_name"], "buses", query={})
set_dataframe(buses_df, "stg_buses", pk_column=None, db_operation="insert", **mysql_args)
print("Load works")


Load works


#### Lets verify that the data is loaded in:

In [53]:
get_sql_dataframe("""
SELECT table_name, table_rows
FROM information_schema.tables
WHERE table_schema = 'transit_dw'
  AND table_name IN ('stg_routes','stg_stops','stg_riders','stg_buses','stg_riders','fact_riders','fact_riders')
ORDER BY table_name;
""", **mysql_args)

,TABLE_NAME,TABLE_ROWS
0,fact_riders,0
1,stg_buses,2
2,stg_riders,3
3,stg_routes,2
4,stg_stops,3


#### To continue with the transformation phase - We will be taking the raw data from the staging files (`stg_routes, stg_buses, stg_stops`) into dimensions and populate them before updating my fact table's foreign keys (Similar to Lab2b)

In [58]:
sql_routes = """
TRUNCATE TABLE dim_routes;
INSERT INTO dim_routes (route_id, route_name, origin, destination, service_type, route_length_km)
SELECT route_id, route_name, origin, destination, service_type, route_length_km
FROM stg_routes;
"""

sql_buses = """
TRUNCATE TABLE dim_buses;
INSERT INTO dim_buses (bus_id, capacity, manufacturer, model, year, fuel_type)
SELECT bus_id, capacity, manufacturer, model, year, fuel_type
FROM stg_buses;
"""

sql_stops = """
TRUNCATE TABLE dim_stops;
INSERT INTO dim_stops (stop_id, stop_name, latitude, longitude, zone)
SELECT stop_id, stop_name, latitude, longitude, zone
FROM stg_stops;
"""
for line in (sql_routes, sql_buses, sql_stops):
    for com in line.split(';'):
        s = com.strip()
        if s:
            connection.execute(text(s))

connection.commit()

print("Dimensions loaded successfully.")


Dimensions loaded successfully.


Let's verify the data was loaded into the dimension tables

In [59]:
get_sql_dataframe("""
SELECT 'dim_routes' AS table_name, COUNT(*) AS `row_count` FROM dim_routes
UNION ALL
SELECT 'dim_buses',  COUNT(*) AS `row_count` FROM dim_buses
UNION ALL
SELECT 'dim_stops',  COUNT(*) AS `row_count` FROM dim_stops;
""", **mysql_args)

,table_name,row_count
0,dim_routes,2
1,dim_buses,2
2,dim_stops,3


#### This step loads the **fact_riders** table with data from the staging table **(stg_riders)**. We clear any old data, then insert new rows that have rider information. After loading,we  update each record with surrogate keys from the dimension tables:
- `date_key` from `dim_date`
- `route_key` from `dim_routes`
- `bus_key` from `dim_buses`
- `stop_key` from `dim_stops`


This links the fact table to all dimensions (riders by route, stop, and time).

In [60]:
fact_sql = """
TRUNCATE TABLE fact_riders;
INSERT INTO fact_riders
(route_id, bus_id, stop_id, ride_dt, time_recorded, riders_count, fare_collected, avg_wait_time)
SELECT route_id, bus_id, stop_id, DATE(ride_dt), time_recorded, riders_count, fare_collected, avg_wait_time
FROM stg_riders;

SET SQL_SAFE_UPDATES = 0;
UPDATE fact_riders AS f
JOIN dim_date AS d ON d.full_date = f.ride_dt
SET f.date_key = d.date_key;

UPDATE fact_riders AS f
JOIN dim_routes AS r ON r.route_id = f.route_id
SET f.route_key = r.route_key;

UPDATE fact_riders AS f
JOIN dim_buses AS b ON b.bus_id = f.bus_id
SET f.bus_key = b.bus_key;

UPDATE fact_riders AS f
JOIN dim_stops AS s ON s.stop_id = f.stop_id
SET f.stop_key = s.stop_key;
"""


for com in fact_sql.split(';'):
    s = com.strip()
    if s:
        connection.execute(text(s))
connection.commit()
print("fact_riders loaded and keys integrated.")

fact_riders loaded and keys integrated.


#### Ensure the data is loaded on fact table:

In [61]:
get_sql_dataframe("SELECT COUNT(*) AS fact_rows FROM fact_riders;", **mysql_args)

,fact_rows
0,3


#### Let's ensure that every record has a valid foreign key that is linked to the dimension tables

In [64]:
get_sql_dataframe("""
SELECT COUNT(*) AS missing_keys
FROM fact_riders
WHERE date_key IS NULL OR route_key IS NULL OR bus_key IS NULL OR stop_key IS NULL;
""", **mysql_args)

,missing_keys
0,0


#### Analytical Queries / Results

In [63]:
get_sql_dataframe("""
SELECT r.route_name, SUM(f.riders_count) AS total_riders
FROM fact_riders f
JOIN dim_routes r ON f.route_key = r.route_key
GROUP BY r.route_name
ORDER BY total_riders DESC;
""", **mysql_args)

,route_name,total_riders
0,Main St Loop,54.0
1,Airport Express,12.0


In [65]:
get_sql_dataframe("""
SELECT d.calendar_year_month, AVG(f.fare_collected) AS avg_fare
FROM fact_riders f
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY d.calendar_year_month
ORDER BY d.calendar_year_month;
""", **mysql_args)

,calendar_year_month,avg_fare
0,2025-09,44.5


In [66]:
get_sql_dataframe("""
SELECT s.stop_name, SUM(f.riders_count) AS total_riders
FROM fact_riders f
JOIN dim_stops s ON f.stop_key = s.stop_key
GROUP BY s.stop_name
ORDER BY total_riders DESC
LIMIT 10;
""", **mysql_args)

,stop_name,total_riders
0,Hospital Entrance,31.0
1,Downtown Hub,23.0
2,Airport Terminal,12.0


In [67]:
get_sql_dataframe("""
SELECT d.calendar_year_month,
       r.route_name,
       SUM(f.riders_count) AS total_riders
FROM fact_riders f
JOIN dim_date d   ON f.date_key = d.date_key
JOIN dim_routes r ON f.route_key = r.route_key
GROUP BY d.calendar_year_month, r.route_name
ORDER BY d.calendar_year_month, total_riders DESC
LIMIT 50;
""", **mysql_args)

,calendar_year_month,route_name,total_riders
0,2025-09,Main St Loop,54.0
1,2025-09,Airport Express,12.0
